# 市场状态识别（Market Regime Detection）—— HMM 实战教程

**目标**：用真实的沪深 300 指数日线，走通"特征工程 → PCA → HMM 状态发现 → 逻辑回归预测 → 可解释性评估"的完整流水线。

**这不是项目代码教程**。本 notebook 自包含，数据 CSV 在 `data/` 目录下，不依赖 binwh.quant-pilot 的前后端。

**依赖**：`pip install numpy pandas matplotlib scikit-learn hmmlearn`

**目录**：
1. 问题：为什么要识别"市场状态"？
2. 两阶段框架（无监督发现 + 监督预测）
3. Step 1 · 读真实数据 + 特征工程
4. Step 2 · PCA 降维（去相关 + 去噪）
5. Step 3 · 选 K（silhouette + HMM 对数似然）
6. Step 4 · HMM 发现隐藏状态
7. Step 5 · 用历史事件验证状态"讲得通"
8. Step 6 · 逻辑回归预测明天的状态
9. Step 7 · ⭐可解释性评估（混淆矩阵 / ROC / 校准曲线 / KS）
10. Step 8 · 完整管线 + 交易决策思路
11. 踩过的坑 & 关键决策

贯穿全篇的问题：**给定沪深 300 指数日线，能不能让模型自己告诉我们"今天是平静还是动荡"？**


## 1. 问题：为什么要识别"市场状态"？

### 朴素做法的问题

如果你只用一个固定规则交易（比如"均线金叉买、死叉卖"），你会发现在不同时期它的表现天差地别：

- **2014–2015 大牛市 + 2015 股灾**：趋势策略先赚翻后被打脸
- **2018 贸易战单边下跌**：趋势策略反复假突破
- **2020 疫情底后强势反弹**：趋势策略赚翻
- **2022–2023 横盘震荡**：趋势策略反复止损

**根本原因**：市场在不同时期运行在"不同的机制（regime）"下。同一个价格信号，在不同 regime 下含义完全不同。

### Market Regime 的核心想法

> 如果我们能先判断"现在市场处于什么状态"，再决定用什么策略，就能避开"一套策略包打天下"的陷阱。

状态是"隐藏"的——我们只看得到价格，看不到"现在到底是哪种状态"。这正是无监督学习的典型场景。

### 为什么不直接用规则（比如波动率 > X 就算动荡）？

- 阈值 X 怎么定？不同时期波动率水平不同，固定阈值不通用
- 单一指标容易被骗（低波动也可能是下跌中继）
- 我们希望模型**自己**从多个特征组合里发现状态，而不是人为硬编码

这就是 **HMM 登场的地方**。


## 2. 两阶段框架（无监督发现 + 监督预测）

> **核心范式**（来自"市场状态识别的混合机器学习实战"）：传统两条路各有短板——只聚类能发现状态但无法预测未来；只分类能预测但没有"真实状态标签"可用。混合框架两阶段打通：

| 阶段 | 方法 | 作用 |
|---|---|---|
| **阶段一（无监督）** | K-Means / GMM / **HMM** 聚类 | 无预定义标签下**发现潜在状态** |
| **阶段二（监督）** | 逻辑回归 / 随机森林 | 把聚类标签当"真值"，**预测未来状态** |

标签由数据自动生成 → 预测模型有了训练目标 → 一举解决两条老路的痛点。

### 为什么本教程用 HMM 而不是 K-Means

- **K-Means 假设每个样本独立**，忽略时间顺序——把日期打乱后结果不变
- **HMM 显式建模时序依赖**："昨天是平静，今天大概率还是平静"——这种持续性正是金融市场的关键特征

### 完整管线

```
原始 OHLCV（沪深 300 + 上证 50 + 中证 500）
  ↓ Step 1: 构造特征（收益率 / 波动率 / 跨盘价差）→ expanding 标准化（防泄漏）
  ↓ Step 2: PCA 降维到 95% 方差
  ↓ Step 3: silhouette 选 K，确定状态数
  ↓ Step 4: HMM 训练 → Viterbi 解码出历史状态序列
  ↓ Step 5: 人工对照 2008 / 2015 / 2020 等历史事件验证
  ↓ Step 6: 逻辑回归预测"明天状态"
  ↓ Step 7: 可解释性评估（混淆矩阵 / ROC / KS / 校准曲线）
交易决策
```


In [ ]:
# 先装依赖（如果还没装）
# !pip install numpy pandas matplotlib scikit-learn hmmlearn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_curve, auc,
    brier_score_loss, roc_auc_score,
)
from sklearn.calibration import calibration_curve
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from hmmlearn.hmm import GaussianHMM

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)
print('环境就绪')

## 3. Step 1 · 读真实数据 + 特征工程

数据在 `data/` 下，沪深 300（主）、上证 50（超大盘）、中证 500（中盘），2010-01 至 2026-07，覆盖 2015 股灾、2018 贸易战、2020 疫情、2022 横盘等多个 regime。

### 特征设计

参考"混合机器学习实战"笔记，构造四类特征：

| 特征族 | 公式 | 含义 |
|---|---|---|
| **多周期收益率** | `close.pct_change(W)` | 短/中/长动量 |
| **已实现波动率** | `log_ret.ewm(W).std() * sqrt(252)` | 年化波动，regime 关键信号 |
| **跨盘价差变化** | `log(中证500) − log(上证50)` 的一阶差分 | 中小盘 vs 超大盘的相对强弱 |
| **成交量变化** | `volume.pct_change(W)` | 量能配合 |

### 防泄漏（灵魂）

标准化时**不能**用全样本均值/方差——那是用未来信息。正确做法：**expanding 窗口**，时刻 t 只用 t 之前的数据。


In [ ]:
DATA_DIR = Path('data')

def load_close(code: str) -> pd.Series:
    df = pd.read_csv(DATA_DIR / f'{code}.csv', parse_dates=['date']).set_index('date').sort_index()
    return df['close'].astype(float)

def load_volume(code: str) -> pd.Series:
    df = pd.read_csv(DATA_DIR / f'{code}.csv', parse_dates=['date']).set_index('date').sort_index()
    return df['volume'].astype(float)

# 主标的：沪深 300
close_300 = load_close('000300')
volume_300 = load_volume('000300')
# 跨盘辅助：上证 50（超大盘）vs 中证 500（中盘）
close_50  = load_close('000016')
close_500 = load_close('000905')

print(f'沪深300: {close_300.shape[0]} 行，{close_300.index[0].date()} ~ {close_300.index[-1].date()}')
print(f'上证50 : {close_50.shape[0]} 行')
print(f'中证500: {close_500.shape[0]} 行')

# 看看价格序列
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(close_300.index, close_300.values, color='steelblue', linewidth=1, label='沪深300')
ax.set_title('沪深 300 收盘价（2010-2026）')
ax.set_ylabel('点位'); ax.legend()
ax.annotate('2015 股灾', xy=(pd.Timestamp('2015-09'), 3200), fontsize=9, color='red')
ax.annotate('2018 贸易战', xy=(pd.Timestamp('2018-07'), 3500), fontsize=9, color='red')
ax.annotate('2020 疫情底', xy=(pd.Timestamp('2020-03'), 3600), fontsize=9, color='red')
plt.tight_layout(); plt.show()

In [ ]:
TRADING_DAYS = 252

def build_features(close, volume, aux_closes=None, aux_names=None):
    """构造无泄漏特征矩阵。aux_closes 用于构造跨资产价差。"""
    feats = pd.DataFrame(index=close.index)
    # 1. 多周期收益率（短中长）
    for w in [1, 5, 21]:
        feats[f'ret_{w}d'] = close.pct_change(w)
    # 2. 已实现波动率（年化，指数加权）
    log_ret = np.log(close.replace([0, np.inf, -np.inf], np.nan)).diff()
    for span in [21, 63]:
        feats[f'realvol_{span}d'] = log_ret.ewm(span=span).std() * np.sqrt(TRADING_DAYS)
    # 3. 成交量变化
    feats['vol_chg_5d'] = volume.pct_change(5).clip(-1, 5)
    # 4. 跨盘价差变化（中证500 − 上证50：中小盘 vs 超大盘相对强弱）
    if aux_closes and len(aux_closes) >= 2:
        a, b = aux_closes[:2]
        # 对齐到主标的交易日
        df = pd.concat([a.rename('a'), b.rename('b')], axis=1).reindex(close.index).ffill()
        spread = np.log(df['a'].replace([0, np.inf, -np.inf], np.nan)) - \
                 np.log(df['b'].replace([0, np.inf, -np.inf], np.nan))
        feats['cross_spread_chg'] = spread.diff()

    # === 防泄漏：expanding 标准化（min_periods=60 ≈ 3 个月 warmup）===
    mean = feats.expanding(min_periods=60).mean()
    std = feats.expanding(min_periods=60).std()
    scaled = ((feats - mean) / std).replace([np.inf, -np.inf], np.nan).dropna()

    # 剔除有效样本不足 50% 的列（保险，通常不会触发）
    min_valid = max(1, int(len(scaled) * 0.5))
    scaled = scaled.dropna(axis=1, thresh=min_valid)
    return scaled

X = build_features(close_300, volume_300, aux_closes=[close_500, close_50])
print(f'特征矩阵: {X.shape[0]} 行 × {X.shape[1]} 列')
print(f'特征列: {list(X.columns)}')
X.tail(3)

## 4. Step 2 · PCA 降维（去相关 + 去噪）

### 为什么需要 PCA

我们刚构造了 7 个特征（3 个收益率 + 2 个波动率 + 1 个量能 + 1 个跨盘价差）。问题：
1. **特征高度相关**：ret_1d / ret_5d / ret_21d 明显相关，直接喂 HMM 会放大噪声
2. **HMM 高维不稳**：高斯协方差矩阵维度高了容易奇异

PCA 把相关特征"压缩"成几个不相关的主成分，保留 95% 方差即可。

### 直觉

把特征空间想象成一片云——PCA 找出点云"最长的几个方向"，把数据投影过去。通常前 2-3 个主成分就能解释绝大部分波动。


In [ ]:
# === PCA 降维（保留 95% 方差）===
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X.values)

print(f'PCA: {X.shape[1]} 维 → {X_pca.shape[1]} 维')
print(f'各主成分方差解释比: {pca.explained_variance_ratio_.round(3)}')
print(f'累计解释: {pca.explained_variance_ratio_.cumsum().round(3)}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_, color='steelblue')
axes[0].set_title('各主成分方差解释比'); axes[0].set_xlabel('主成分'); axes[0].set_ylabel('方差占比')

y2 = X_pca[:, 1] if X_pca.shape[1] > 1 else np.zeros(len(X_pca))
axes[1].scatter(X_pca[:, 0], y2, c=range(len(X_pca)), cmap='viridis', s=6, alpha=0.4)
axes[1].set_xlabel('主成分 1'); axes[1].set_ylabel('主成分 2' if X_pca.shape[1] > 1 else '(仅1个主成分)')
axes[1].set_title('PCA 空间（颜色=时间顺序，早期=紫，后期=黄）')
plt.tight_layout(); plt.show()

## 5. Step 3 · 选 K（状态数）

### K 怎么定

没有银弹，但有几个工具辅助：
- **silhouette 分数**（KMeans + 聚类紧密度，越接近 +1 越好）——看 K=2/3/4 哪个分数最高
- **HMM 对数似然**（不同状态数下，模型对数据的拟合好坏）
- **可解释性**（K=3 时第 3 个状态讲不讲得通？）

常见选择：
- **K=2**：平静 vs 动荡（最常用、最稳、最好解释）
- **K=3**：牛市 / 熊市 / 震荡（更细，但训练不稳）
- **K=4+**：很少用，容易过拟合


In [ ]:
# === silhouette 选 K（KMeans 作为快速代理）===
sil_scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(X_pca)
    sil_scores[k] = silhouette_score(X_pca, km.labels_)

print('K → silhouette 分数（越接近 1 越好）:')
for k, s in sil_scores.items():
    bar = '█' * int(s * 50)
    print(f'  K={k}: {s:.3f}  {bar}')

best_k_sil = max(sil_scores, key=sil_scores.get)
print(f'\n按 silhouette 最优: K={best_k_sil}')
print(f'→ 但实务里优先选 K=2，原因：可解释性最强、训练最稳、HMM 持续性假设最贴合')

In [ ]:
# === HMM 对数似然比较 K=2 / 3 / 4 ===
hmm_scores = {}
for k in [2, 3, 4]:
    try:
        h = GaussianHMM(n_components=k, covariance_type='full', n_iter=100, random_state=42)
        h.fit(X_pca)
        # 用平均对数似然（除以样本数）方便比较
        score = h.score(X_pca) / len(X_pca)
        hmm_scores[k] = score
        print(f'  K={k}: 平均对数似然 = {score:.3f}')
    except Exception as e:
        print(f'  K={k}: 失败 ({e})')

print('\n注意：对数似然随 K 单调上升是常态（模型更复杂总能拟合更好），')
print('不能只看它选 K——要结合 silhouette、可解释性、AIC/BIC 综合判断。')
print('实务建议：从 K=2 起步，确认能识别出"平静 vs 动荡"再考虑加状态。')

## 6. Step 4 · HMM 发现隐藏状态

### HMM 训练在干什么

`GaussianHMM(n_components=2)` 告诉模型：假设有 2 个状态，自己去学：
- 每个状态下观测值的高斯分布（均值向量 + 协方差矩阵）
- 状态间转移概率矩阵（2×2）
- 初始状态分布

学习算法叫 **Baum-Welch**（EM 算法的变种）：先猜一组参数 → 用这组参数算每个时刻最可能的状态 → 用状态重新更新参数 → 迭代到收敛。

训练完后用 **Viterbi 算法**解码：返回"最可能的状态序列"。

### 多 seed 抗局部最优

Baum-Welch 依赖初始化，容易陷局部最优。我们训 10 次取对数似然最大的那组。


In [ ]:
# === HMM 训练 + 解码（多 seed 取最优）===
N_STATES = 2
best_hmm = None
best_score = -np.inf
for seed in range(10):
    h = GaussianHMM(n_components=N_STATES, covariance_type='full', n_iter=100, random_state=seed)
    try:
        h.fit(X_pca)
        s = h.score(X_pca)
        if s > best_score:
            best_score = s
            best_hmm = h
    except Exception:
        continue

hmm = best_hmm
predicted_states = hmm.predict(X_pca)
state_probs = hmm.predict_proba(X_pca)

print(f'最优 seed 的对数似然: {best_score:.1f}')
print(f'\n转移概率矩阵:\n{hmm.transmat_.round(3)}')
print(f'\n状态持续性解读:')
print(f'  状态0→状态0: {hmm.transmat_[0,0]:.1%}（一旦进入状态0，平均停留 {1/(1-hmm.transmat_[0,0]):.0f} 天）')
print(f'  状态1→状态1: {hmm.transmat_[1,1]:.1%}（一旦进入状态1，平均停留 {1/(1-hmm.transmat_[1,1]):.0f} 天）')

In [ ]:
# === 状态对齐 + 可视化 ===
# HMM 的状态编号是任意的，要"对齐"——波动率高的那个定义为"动荡"
state_vol = {s: np.std(X.loc[X.index[i], 'realvol_21d']) for i, s in enumerate(predicted_states)} if False else None
# 更稳的做法：直接看每个状态对应的原始波动率均值
vol_by_state = pd.Series(predicted_states, index=X.index).to_frame('state').assign(
    realvol=X['realvol_21d']
).groupby('state')['realvol'].mean()
print('每个状态的原始波动率均值:')
print(vol_by_state)
# 波动率高的就是"动荡"
if vol_by_state.iloc[0] > vol_by_state.iloc[1]:
    state_map = {0: 1, 1: 0}  # 翻转
else:
    state_map = {0: 0, 1: 1}
aligned_states = pd.Series([state_map[s] for s in predicted_states], index=X.index)
print(f'\n状态对齐: {state_map}（0=平静，1=动荡）')

In [ ]:
# === 可视化：价格 + 状态色带 ===
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
close_aligned = close_300.reindex(X.index)
axes[0].plot(close_aligned.index, close_aligned.values, color='steelblue', linewidth=0.8, label='沪深300')
# 用 fill_between 把动荡期标红
for i in range(len(aligned_states)):
    if aligned_states.iloc[i] == 1 and (i == 0 or aligned_states.iloc[i-1] == 0):
        # 动荡期开始
        j = i
        while j < len(aligned_states) and aligned_states.iloc[j] == 1:
            j += 1
        axes[0].axvspan(aligned_states.index[i], aligned_states.index[j-1],
                        color='salmon', alpha=0.3, label='动荡期' if i < 10 else '')
axes[0].set_title('沪深 300 收盘价 + HMM 识别的动荡期（红色阴影）')
axes[0].set_ylabel('点位'); axes[0].legend(loc='upper left')

axes[1].fill_between(aligned_states.index, aligned_states.values, color='purple', alpha=0.4, step='pre')
axes[1].set_yticks([0, 1]); axes[1].set_ylabel('状态')
axes[1].set_yticklabels(['平静', '动荡'])
plt.tight_layout(); plt.show()

## 7. Step 5 · 用历史事件验证状态"讲得通"

HMM 输出的 0/1 本身没有意义——人工对照历史事件是**把数字变成可解释知识的唯一方式**。

### 统计画像

对每个状态算：平均收益、波动率、最大回撤、占比。如果"动荡"状态波动率是平静的 2-3 倍、收益为负——那就很有信心说它就是熊市/动荡。


In [ ]:
# === 状态统计画像 ===
ret_300 = close_300.pct_change().reindex(X.index)
state_df = pd.DataFrame({'state': aligned_states, 'ret': ret_300})

print('状态画像:')
print(f'{"状态":<10} {"日均收益":>12} {"年化波动":>10} {"最大回撤":>10} {"天数":>6} {"占比":>8}')
for s in [0, 1]:
    sub = state_df[state_df['state'] == s]['ret']
    label = '平静' if s == 0 else '动荡'
    cum = (1 + sub).cumprod()
    peak = cum.cummax()
    mdd = ((cum - peak) / peak).min()
    print(f'{label:<10} {sub.mean():>12.4%} {sub.std()*np.sqrt(252):>10.2%} {mdd:>10.2%} {len(sub):>6} {len(sub)/len(state_df):>8.1%}')

print('\n→ 如果动荡期波动率 ≈ 平静期的 2-3 倍 + 最大回撤显著更深，就说明 HMM 真的学到了"regime"。')

In [ ]:
# === 对照已知重大事件 ===
# 注意：HMM 不知道这些事件，是我们事后用来验证标签的合理性
events = [
    ('2015 股灾',     '2015-06', '2015-09'),
    ('2018 贸易战',   '2018-03', '2018-12'),
    ('2020 疫情底',   '2020-02', '2020-04'),
    ('2024 9·24 反弹','2024-09', '2024-10'),
]
print('重大事件期间 HMM 给出的状态分布:')
print(f'{"事件":<18} {"动荡天数占比":>12} {"判断":<8}')
for name, start, end in events:
    mask = (aligned_states.index >= start) & (aligned_states.index <= end)
    if mask.sum() == 0:
        print(f'{name:<18} {"无数据":>12}')
        continue
    turmoil_pct = aligned_states[mask].mean()
    judge = '动荡 ✓' if turmoil_pct > 0.5 else '平静 ✗'
    print(f'{name:<18} {turmoil_pct:>12.1%} {judge}')

print('\n→ 股灾/疫情底应该以动荡为主；如果判断错了，说明特征工程或状态数要调整。')

## 8. Step 6 · 逻辑回归预测明天的状态

### 为什么还要加一层监督学习

HMM 给的是**事后**状态（Viterbi 用整段历史解码），不能直接预测。
交易场景需要：**今天收盘后，预测明天处于什么状态**。

### 思路

把 HMM 输出的状态标签当"真值"，训练逻辑回归：
- **输入**：今天的特征（或主成分）
- **输出**：明天处于状态 0/1 的概率

### 时序切分（关键）

绝不能随机分训练/测试集——那会用未来训练过去。必须**前向切分**：前 80% 训练，后 20% 测试。


In [ ]:
# === 训练逻辑回归预测"明天的状态" ===
y_tomorrow = aligned_states.shift(-1).dropna()
X_today = pd.DataFrame(X_pca, index=X.index, columns=[f'pc{i+1}' for i in range(X_pca.shape[1])])
common_idx = y_tomorrow.index.intersection(X_today.index)
X_today = X_today.loc[common_idx]
y_tomorrow = y_tomorrow.loc[common_idx]

# 时序切分：前 80% 训练，后 20% 测试（绝不 shuffle！）
split = int(len(common_idx) * 0.8)
split_date = common_idx[split]
X_train, X_test = X_today.iloc[:split], X_today.iloc[split:]
y_train, y_test = y_tomorrow.iloc[:split], y_tomorrow.iloc[split:]

clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
print(f'切分日期: {split_date.date()}（训练集 {len(y_train)} 行，测试集 {len(y_test)} 行）')
print(f'\n测试集准确率: {acc:.1%}')
print(f'\n逻辑回归系数: {dict(zip(X_today.columns, clf.coef_[0].round(3)))}')
print(f'→ 正系数 = 该主成分大时更可能动荡；负系数 = 更可能平静。')

## 9. Step 7 · ⭐可解释性评估（XAI 兜底）

**光有高准确率不够，要让模型"可解释、可信赖"**。这是"混合机器学习实战"笔记里的核心环节，用了 6 种工具：

| 工具 | 回答的问题 |
|---|---|
| **混淆矩阵** | 错分发生在哪类→哪类？ |
| **ROC + AUC** | 整体排序能力有多强？ |
| **KS 统计量** | 两类概率分布最大分离点在哪？ |
| **校准曲线** | 预测概率 0.7 是不是真的 70% 发生？ |
| **Brier 分数** | 概率预测均方误差（越低越好） |
| **判别阈值** | 默认 0.5 一定最优吗？

这一步是"相信模型"的最后防线——高准确率可能是样本失衡的假象，KS=0.9+ 才是真分离。


In [ ]:
# === 1. 混淆矩阵 ===
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 混淆矩阵
ax = axes[0, 0]
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)
ax.set_xticks([0, 1]); ax.set_xticklabels(['预测平静', '预测动荡'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['实际平静', '实际动荡'])
ax.set_title(f'混淆矩阵（准确率 {acc:.1%}）')
plt.colorbar(im, ax=ax)

# === 2. ROC + AUC ===
ax = axes[0, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='随机')
ax.set_xlabel('假阳率'); ax.set_ylabel('真阳率')
ax.set_title('ROC 曲线'); ax.legend(loc='lower right'); ax.grid(alpha=0.3)

# === 3. 两类概率分布 + KS 统计量 ===
ax = axes[1, 0]
proba_calm = y_proba[y_test == 0]
proba_turm = y_proba[y_test == 1]
ax.hist(proba_calm, bins=30, alpha=0.5, color='green', label='实际平静', density=True)
ax.hist(proba_turm, bins=30, alpha=0.5, color='red', label='实际动荡', density=True)
# KS 统计量
from scipy.stats import ks_2samp
ks_stat, ks_p = ks_2samp(proba_calm, proba_turm)
ax.set_title(f'预测概率分布（KS = {ks_stat:.3f}, p < 0.001）')
ax.set_xlabel('预测动荡概率'); ax.set_ylabel('密度'); ax.legend()
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='默认阈值 0.5')

# === 4. 校准曲线 + Brier ===
ax = axes[1, 1]
frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy='quantile')
brier = brier_score_loss(y_test, y_proba)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='完美校准')
ax.plot(mean_pred, frac_pos, 's-', color='darkorange', label=f'模型（Brier = {brier:.4f}）')
ax.set_xlabel('预测概率均值'); ax.set_ylabel('实际发生比例')
ax.set_title('校准曲线'); ax.legend(loc='upper left'); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f'\n评估小结:')
print(f'  准确率   : {acc:.3f}')
print(f'  ROC-AUC  : {roc_auc:.3f}')
print(f'  KS 统计量: {ks_stat:.3f}（> 0.4 算良好分离，> 0.7 算强分离）')
print(f'  Brier 分数: {brier:.4f}（越低越好，0 = 完美，0.25 = 随机猜）')

In [ ]:
# === 5. 判别阈值分析（找最佳阈值）===
thresholds = np.linspace(0.1, 0.9, 81)
f1s, precs, recs = [], [], []
for t in thresholds:
    yp = (y_proba >= t).astype(int)
    tp = ((yp == 1) & (y_test == 1)).sum()
    fp = ((yp == 1) & (y_test == 0)).sum()
    fn = ((yp == 0) & (y_test == 1)).sum()
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0
    f1s.append(f); precs.append(p); recs.append(r)

best_idx = int(np.argmax(f1s))
best_t = thresholds[best_idx]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1s, label='F1', color='darkorange', lw=2)
ax.plot(thresholds, precs, label='精确率', color='steelblue', alpha=0.7)
ax.plot(thresholds, recs, label='召回率', color='green', alpha=0.7)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='默认阈值 0.5')
ax.axvline(best_t, color='red', linestyle='--', alpha=0.7, label=f'最佳 F1 阈值 {best_t:.2f}')
ax.set_xlabel('判别阈值'); ax.set_ylabel('分数'); ax.set_title('判别阈值分析')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'默认阈值 0.5 的 F1: {f1s[40]:.3f}')
print(f'最佳 F1 阈值: {best_t:.2f}（F1 = {f1s[best_idx]:.3f}）')
print(f'→ 如果你更怕漏报动荡（规避大跌），可以降低阈值到 0.3 左右提升召回率。')

## 10. Step 8 · 完整管线 + 交易决策思路

### 怎么把状态概率变成交易决策

几种常见策略（不是建议，只是示例）：

**策略 A · 状态切换减仓**
```
if 明天动荡概率 > 60%:
    降低仓位到 30%
else:
    保持满仓
```

**策略 B · 状态相关的策略切换**
```
if 平静期:
    用趋势策略（均线、动量）
elif 动荡期:
    用均值回复策略（布林带、配对交易）
```

**策略 C · 概率加权仓位**
```
仓位 = 1 - 动荡概率
# 概率 50% 时仓位 50%，平滑过渡，避免频繁切换
```

**重要提醒**：状态识别本身不赚钱，它只是"环境判断器"。真正赚钱的是基于状态切换的策略——这需要单独回测验证。**不要**看到准确率 90% 就以为能赚钱。


## 11. 踩过的坑 & 关键决策

### 坑 1：数据泄漏（最致命）

**错误做法**：用全样本均值/方差做标准化 → 等于告诉模型"未来长这样" → 训练准确率虚高，实盘立刻崩。

**正确做法**：expanding 窗口，时刻 t 只用 t 之前的数据。同理，滚动特征窗口不能跨越当前时刻。

### 坑 2：HMM 的局部最优

Baum-Welch 是 EM 算法，依赖初始化，容易陷局部最优。
**解决**：多 seed 训练取对数似然最大的（本教程训 10 次取最优）。

### 坑 3：状态编号是任意的

HMM 输出 0/1，但哪个是"平静"是随机的——每次重新训练都可能互换。
**解决**：训练后用"波动率排序"对齐——波动率高的定义为动荡。

### 坑 4：PCA 必须在训练集上 fit，测试集 transform

全样本 fit PCA 会有轻微泄漏。严格前向验证时要分批 fit。

### 坑 5：不要被准确率骗

状态识别 90% 准确率可能只是因为"状态持续性很强"——模型一直猜"今天 = 昨天"就能达到 85%+。判断状态识别质量要用：
- **KS 统计量**：两类概率分布是否显著分离（> 0.4 良好，> 0.7 强）
- **ROC-AUC**：整体排序能力（> 0.8 良好）
- **Silhouette**：主成分空间状态是否分开
- **事件覆盖率**：已知重大事件期间是否正确标记为动荡

### 坑 6：随机切分训练/测试是禁忌

`train_test_split(shuffle=True)` 在时序数据上是数据泄漏——测试集可能从训练集中间抽，模型"见过未来"。必须**前向切分**。

---

## 对照"混合机器学习实战"笔记

本教程对照原笔记的实现情况：

| 笔记方法 | 本教程实现 |
|---|---|
| 两阶段框架（无监督 + 监督） | ✅ HMM + 逻辑回归 |
| 信用利差 / 多资产特征 | ✅ 改造为 A 股跨盘价差（中证500 − 上证50） |
| 多周期收益率 + 实现波动率 | ✅ |
| expanding 标准化防泄漏 | ✅ 且更严格（min_periods=60） |
| PCA 降维 | ✅ 保留 95% 方差 |
| silhouette 选 K | ✅ Step 3 |
| K-Means 聚类 | ⬆️ 升级为 HMM（显式建模时序持续性） |
| 逻辑回归分类 | ✅ 且更严谨（时序切分而非随机分层） |
| **混淆矩阵** | ✅ |
| **ROC-AUC** | ✅ |
| **KS 统计量** | ✅ |
| **校准曲线 + Brier** | ✅ |
| **判别阈值分析** | ✅ |
| **学习曲线** | ⏸️ 留作练习（`sklearn.model_selection.learning_curve`） |

---

**核心 takeaway**：市场状态识别 = 特征工程（无泄漏）+ HMM（发现隐藏状态）+ 人工验证（讲得通）+ 监督学习（预测明天）+ 可解释性评估（值得信赖）。没有任何一步是黑魔法，每一步都有明确的思路和可检查的中间产物。
